# 🧪 Direct Hugging Face Transformers API for DeepSeek-V4-Pro (Qwen3.5-9B) Uncensored

This notebook runs the original safetensors repository [`xtools-at/DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic`](https://huggingface.co/xtools-at/DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic) directly via PyTorch **Transformers + BitsAndBytes 4-bit (NF4)** on Kaggle Free GPU.

Features:
- 🎯 Direct safetensors loading from Hugging Face.
- 📉 4-bit NF4 Quantization (fits cleanly in ~6.5GB VRAM on Kaggle T4 or P100).
- 🌐 Cloudflare Free Quick Tunnel for public HTTPS access.
- 🔒 API Key Authentication guard.
- ⚡ OpenAI-compatible `/v1/chat/completions` with streaming SSE support.

### Step 1: Check GPU Accelerator

In [ ]:
!nvidia-smi

### Step 2: Install Libraries & Cloudflare Tunnel

In [ ]:
%%bash
pip install -q --upgrade pip
pip install -q torch transformers accelerate bitsandbytes fastapi uvicorn[standard] httpx pydantic requests openai

if ! command -v cloudflared &> /dev/null; then
  curl -sL --output /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
  chmod +x /usr/local/bin/cloudflared
fi
echo "[✓] Cloudflared ready: $(cloudflared --version)"

### Step 3: Load Model with BitsAndBytes 4-Bit Quantization

In [ ]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "xtools-at/DeepSeek-V4-Pro-Qwen3.5-9B-UNCENSORED-heretic"
API_KEY = "sk-deepseek-uncensored-kaggle-key"
os.environ["API_KEY"] = API_KEY
PORT = 8000

print(f"[*] Loading tokenizer for {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    bnb_4bit_use_double_quant=True
)

print("[*] Loading model weights into GPU (4-bit NF4)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)
print("[✓] Model successfully loaded into GPU VRAM!")

### Step 4: Build FastAPI Server with Streaming & API Key

In [ ]:
import time, json, uuid, threading
from typing import List, Optional, Union, Dict, Any
from fastapi import FastAPI, HTTPException, Header, Query, Depends
from fastapi.responses import StreamingResponse, JSONResponse
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from transformers import TextIteratorStreamer

app = FastAPI(title="DeepSeek-V4-Pro Transformers API")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True, allow_methods=["*"], allow_headers=["*"])

class ChatMessage(BaseModel):
    role: str
    content: str

class ChatRequest(BaseModel):
    model: Optional[str] = MODEL_ID
    messages: List[ChatMessage]
    temperature: Optional[float] = 0.7
    top_p: Optional[float] = 0.9
    max_tokens: Optional[int] = 2048
    stream: Optional[bool] = False

def verify_key(authorization: Optional[str] = Header(None), x_api_key: Optional[str] = Header(None)):
    key = None
    if authorization and authorization.lower().startswith("bearer "):
        key = authorization[7:].strip()
    elif x_api_key:
        key = x_api_key.strip()
    if key != API_KEY:
        raise HTTPException(status_code=401, detail="Unauthorized API Key")
    return key

@app.get("/health")
async def health():
    return {"status": "healthy", "model": MODEL_ID, "engine": "transformers_4bit"}

@app.get("/v1/models")
async def list_models(k: str = Depends(verify_key)):
    return {"object": "list", "data": [{"id": MODEL_ID, "object": "model"}]}

@app.post("/v1/chat/completions")
async def chat(req: ChatRequest, k: str = Depends(verify_key)):
    # Apply chat template
    raw_messages = [{"role": m.role, "content": m.content} for m in req.messages]
    prompt_text = tokenizer.apply_chat_template(raw_messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
    request_id = f"chatcmpl-{uuid.uuid4().hex[:12]}"
    created_ts = int(time.time())

    if req.stream:
        streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
        gen_kwargs = dict(
            **inputs,
            streamer=streamer,
            max_new_tokens=req.max_tokens,
            temperature=max(req.temperature, 0.01),
            top_p=req.top_p,
            do_sample=(req.temperature > 0)
        )
        t = threading.Thread(target=model.generate, kwargs=gen_kwargs)
        t.start()

        def event_stream():
            for new_text in streamer:
                chunk = {
                    "id": request_id,
                    "object": "chat.completion.chunk",
                    "created": created_ts,
                    "model": MODEL_ID,
                    "choices": [{"index": 0, "delta": {"content": new_text}, "finish_reason": None}]
                }
                yield f"data: {json.dumps(chunk)}\n\n"
            final_chunk = {
                "id": request_id,
                "object": "chat.completion.chunk",
                "created": created_ts,
                "model": MODEL_ID,
                "choices": [{"index": 0, "delta": {}, "finish_reason": "stop"}]
            }
            yield f"data: {json.dumps(final_chunk)}\n\n"
            yield "data: [DONE]\n\n"

        return StreamingResponse(event_stream(), media_type="text/event-stream")
    else:
        outputs = model.generate(
            **inputs,
            max_new_tokens=req.max_tokens,
            temperature=max(req.temperature, 0.01),
            top_p=req.top_p,
            do_sample=(req.temperature > 0)
        )
        generated_ids = outputs[0][inputs.input_ids.shape[1]:]
        response_text = tokenizer.decode(generated_ids, skip_special_tokens=True)
        return {
            "id": request_id,
            "object": "chat.completion",
            "created": created_ts,
            "model": MODEL_ID,
            "choices": [{"index": 0, "message": {"role": "assistant", "content": response_text}, "finish_reason": "stop"}]
        }

### Step 5: Start Server & Cloudflare Tunnel

In [ ]:
import uvicorn, threading, subprocess, re, time

# Run Uvicorn in background thread
server_thread = threading.Thread(target=lambda: uvicorn.run(app, host="127.0.0.1", port=PORT, log_level="warning"), daemon=True)
server_thread.start()
time.sleep(2)

# Start Cloudflare tunnel
tunnel_log = open("tunnel.log", "w")
tunnel_proc = subprocess.Popen(["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{PORT}", "--no-autoupdate"], stdout=tunnel_log, stderr=subprocess.STDOUT)

public_url = None
for _ in range(30):
    time.sleep(1)
    with open("tunnel.log", "r") as f:
        content = f.read()
        match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", content)
        if match:
            public_url = match.group(0)
            break

print("="*70)
print(f" 🌐 Public Base URL: {public_url}/v1")
print(f" 🔑 API Key        : {API_KEY}")
print(f" 📦 Model Name     : {MODEL_ID}")
print("="*70)

### Step 6: Keep Alive Loop

In [ ]:
from datetime import datetime
try:
    while True:
        print(f"[{datetime.now().strftime('%H:%M:%S')}] 🟢 Serving requests at {public_url}/v1", flush=True)
        time.sleep(60)
except KeyboardInterrupt:
    print("Stopped.")